# HR Data: IN to OUT
Reads every data product file from the `IN` volume folder (e.g. `Company_2026-11-03_500.csv`), extracts the datephoto (`YYYY-MM-DD_SSS`) from the file name, and writes the file to `OUT/YYYY/MM/DD/` with the datephoto removed from the name (e.g. `OUT/2026/11/03/Company.csv`).

* `mode` = `copy` keeps the source files in `IN`; `move` removes them after a successful copy.
* If several files of the same product exist for the same day (different `SSS`), the one with the highest `SSS` is written last and wins.
* Files that do not match the naming pattern are skipped and reported.

In [0]:
dbutils.widgets.text("in_path", "/Volumes/team_2/source/hr_data/IN/")
dbutils.widgets.text("out_path", "/Volumes/team_2/source/hr_data/OUT/")
dbutils.widgets.dropdown("mode", "copy", ["copy", "move"])
dbutils.widgets.dropdown("overwrite", "true", ["true", "false"])

IN_PATH = dbutils.widgets.get("in_path").rstrip("/")
OUT_PATH = dbutils.widgets.get("out_path").rstrip("/")
MODE = dbutils.widgets.get("mode")
OVERWRITE = dbutils.widgets.get("overwrite") == "true"

print(f"IN: {IN_PATH}\nOUT: {OUT_PATH}\nMODE: {MODE}\nOVERWRITE: {OVERWRITE}")

In [0]:
import re
from pyspark.sql import functions as F

# <product>_<YYYY-MM-DD>_<SSS>.<ext>   e.g. Company_2026-11-03_500.csv
PATTERN = re.compile(r"^(?P<product>.+?)_(?P<yyyy>\d{4})-(?P<mm>\d{2})-(?P<dd>\d{2})_(?P<sss>\d{3})(?P<ext>\.[^.]+)?$")

rows, skipped = [], []
for f in dbutils.fs.ls(IN_PATH):
    if f.isDir():
        continue
    m = PATTERN.match(f.name)
    if not m:
        skipped.append(f.name)
        continue
    g = m.groupdict()
    ext = g["ext"] or ""
    target_dir = f"{OUT_PATH}/{g['yyyy']}/{g['mm']}/{g['dd']}"
    rows.append((f.path, f.name, g["product"], f"{g['yyyy']}-{g['mm']}-{g['dd']}", g["sss"],
                 target_dir, f"{target_dir}/{g['product']}{ext}"))

files_df = spark.createDataFrame(
    rows, "source_path string, source_name string, product string, file_date string, sss string, target_dir string, target_path string"
).withColumn("file_date", F.to_date("file_date"))

print(f"Matched files: {len(rows)} | Skipped (pattern mismatch): {len(skipped)}")
if skipped:
    print("Skipped:", skipped)
display(files_df.orderBy("product", "file_date", "sss"))

In [0]:
results = []
# Sorted so that, for the same product and day, the highest SSS is written last and wins
for r in sorted(files_df.collect(), key=lambda r: (r.product, r.file_date, r.sss)):
    try:
        if not OVERWRITE:
            try:
                dbutils.fs.ls(r.target_path)
                results.append((r.source_name, r.target_path, "SKIPPED_EXISTS", None))
                continue
            except Exception:
                pass  # target does not exist
        dbutils.fs.mkdirs(r.target_dir)
        dbutils.fs.cp(r.source_path, r.target_path)
        if MODE == "move":
            dbutils.fs.rm(r.source_path)
        results.append((r.source_name, r.target_path, "OK", None))
    except Exception as e:
        results.append((r.source_name, r.target_path, "FAILED", str(e)[:500]))

results_df = spark.createDataFrame(results, "source_name string, target_path string, status string, error string")
display(results_df.groupBy("status").count())
display(results_df.orderBy("target_path"))

failed = results_df.filter("status = 'FAILED'").count()
if failed:
    raise Exception(f"{failed} file(s) failed to process - see results above")

In [0]:
# Row-count check: source vs target for each processed file
def row_count(path):
    return spark.read.option("header", "true").csv(path).count()

checks = [(r.source_name, r.target_path, row_count(r.target_path))
          for r in results_df.filter("status = 'OK'").collect()]
display(spark.createDataFrame(checks, "source_name string, target_path string, target_rows long"))

# Preview a sample written file
if checks:
    display(spark.read.option("header", "true").csv(checks[0][1]).limit(10))